# Auto benchmark analysis

In [2]:
import math
import os
import json5
import prettytable
import numpy as np

# Directory Management
try:
    # Run in Terminal
    ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
except:
    # Run in ipykernel & interactive
    ROOT_DIR = os.getcwd()

class OptBenchmarkAnalysis:
    def __init__(self, benchmark_result : dict):
        self.benchmark_result = benchmark_result
    
    def print_table(self, variable : str):
        planner_num = len(self.benchmark_result)
        planner_names = list(self.benchmark_result.keys())
        demo_names = list(self.benchmark_result[planner_names[0]].keys())
        demo_num = len(list(self.benchmark_result.values())[0])
        # create array
        results = np.zeros((planner_num, demo_num))
        for planner, demos in self.benchmark_result.items():
            for demo, benchmarks in demos.items():
                for benchmark, value in benchmarks.items():
                    if benchmark == variable:
                        results[list(self.benchmark_result.keys()).index(planner), list(demos.keys()).index(demo)] = value
        
        # create table
        results = results.tolist()
        table = prettytable.PrettyTable()
        table.field_names =["planner-"+variable]+ demo_names
        for i in range(planner_num):
            table.add_row([planner_names[i]] + [results[i][j] for j in range(demo_num)])
        # set table display precision
        table.float_format = ".3"
        print(table)
        
    def get_summary(self, csv_mode = False):
        planner_num = len(self.benchmark_result)
        planner_names = list(self.benchmark_result.keys())
        demo_names = list(self.benchmark_result[planner_names[0]].keys())
        demo_num = len(list(self.benchmark_result.values())[0])
        table = prettytable.PrettyTable()
        table.field_names = ["Planner", "Time(ms)", "Len.(rad)", "Ctrl.", "Succ. Num.", "Succ. Rate"]
        planner_names_correct = {"flt_cfg_planner_fast":"Proposed(PITD)",
                                 "rrt_cfg_planner":"RRT-Connect",
                                 "rrt_planner":"RRT-Connect (E3)",
                                 "minco_cfg_planner":"MINCO+LBFGS",
                                 "stomp_cfg_planner":"STOMP",
                                 "fec_planner":"FEC",}
        for planner, demos in self.benchmark_result.items():
            tot_optnum = 0
            tot_successnum = 0
            
            tot_time = 0
            tot_time2 = 0
            min_time = math.inf
            max_time = 0
            std_time = 0
            
            tot_len = 0
            tot_len2 = 0
            min_len = math.inf
            max_len = 0
            std_len = 0
            
            tot_ctrl = 0
            tot_ctrl2 = 0
            min_ctrl = math.inf
            max_ctrl = 0
            std_ctrl = 0
            
            for demo, benchmarks in demos.items():
                tot_optnum += benchmarks["OptNum"]
                tot_successnum += benchmarks["OptNum"] * benchmarks["SuccessRate"]
                # Time calculation
                tot_time += benchmarks["Totaltime"]
                tot_time2 += benchmarks["AveTime2"] * benchmarks["OptNum"]
                if benchmarks["MinTime"] < min_time and benchmarks["MinTime"] != 0:
                    min_time = benchmarks["MinTime"]
                if benchmarks["MaxTime"] > max_time:
                    max_time = benchmarks["MaxTime"]
                # Length calculation
                tot_len += benchmarks["AveLen"] * benchmarks["SuccessNum"]
                tot_len2 += benchmarks["AveLen2"] * benchmarks["SuccessNum"]
                if benchmarks["MinLen"] < min_len and benchmarks["MinLen"] != 0:
                    min_len = benchmarks["MinLen"]
                if benchmarks["MaxLen"] > max_len:
                    max_len = benchmarks["MaxLen"]
                # Control calculation
                tot_ctrl += benchmarks["AveCtrl"] * benchmarks["SuccessNum"]
                tot_ctrl2 += benchmarks["AveCtrl2"] * benchmarks["SuccessNum"]
                if benchmarks["MinCtrl"] < min_ctrl and benchmarks["MinCtrl"] != 0:
                    min_ctrl = benchmarks["MinCtrl"]
                if benchmarks["MaxCtrl"] > max_ctrl:
                    max_ctrl = benchmarks["MaxCtrl"]
            ave_successrate = tot_successnum / tot_optnum
            # Time calculation
            ave_time = tot_time / tot_optnum
            # std = sqrt(expectation of square - square of expectation)
            std_time = math.sqrt(max(tot_time2 / tot_optnum - ave_time**2, 0))
            # Length calculation
            ave_len = tot_len / tot_successnum
            std_len = math.sqrt(max(tot_len2 / tot_successnum - ave_len**2, 0))
            # Control calculation
            ave_ctrl = tot_ctrl / tot_successnum
            std_ctrl = math.sqrt(max(tot_ctrl2 / tot_successnum - ave_ctrl**2, 0))
            
            table.add_row([planner_names_correct[planner]+"-mean", ave_time, ave_len, ave_ctrl, tot_successnum, ave_successrate] )
            table.add_row([planner_names_correct[planner]+"-max", max_time, max_len, max_ctrl, tot_successnum, ave_successrate])
            table.add_row([planner_names_correct[planner]+"-std", std_time, std_len, std_ctrl, tot_successnum, ave_successrate])
            if not csv_mode:
                table.add_row(["", "", "", "", "", ""])
        table.float_format = ".3"
        return table
    
    def print_summary(self):
        print(self.get_summary())
        
    def save_summary_csv(self, filename : str):
        table = self.get_summary(csv_mode=True)
        table.float_format = ".3"
        with open(filename, "w") as f:
            string = table.get_csv_string()
            string = string.replace("\n", "")
            f.write(string)

# Reachable Benchmark Analysis            
class ReachableBenchmarkAnalysis(object):
    def __init__(self, benchmark_result : dict, reachable_array : dict):
        self.benchmark_result = benchmark_result
        self.reachable_array = reachable_array
    
    def print_table(self, variable : str):
        planner_num = len(self.benchmark_result)
        planner_names = list(self.benchmark_result.keys())
        demo_names = list(self.benchmark_result[planner_names[0]].keys())
        demo_num = len(list(self.benchmark_result.values())[0])
        # create array
        results = np.zeros((planner_num, demo_num))
        for planner, demos in self.benchmark_result.items():
            for demo, benchmarks in demos.items():
                for benchmark, value in benchmarks.items():
                    if benchmark == variable:
                        results[list(self.benchmark_result.keys()).index(planner), list(demos.keys()).index(demo)] = value
        
        # create table
        results = results.tolist()
        table = prettytable.PrettyTable()
        table.field_names =["planner-"+variable]+ demo_names
        for i in range(planner_num):
            table.add_row([planner_names[i]] + [results[i][j] for j in range(demo_num)])
        # set table display precision
        table.float_format = ".3"
        print(table)
        
    def analysis_reachable(self, simplified = True):
        planner_num = len(self.benchmark_result)
        planner_names = list(self.benchmark_result.keys())
        demo_names = list(self.benchmark_result[planner_names[0]].keys())
        demo_num = len(list(self.benchmark_result.values())[0])
        table = prettytable.PrettyTable()
        # table.field_names = ["Planner", "Accuracy", "Precision", "Recall", "F1"]
        if simplified:
            table.field_names = ["Planner", "Ave.Time(ms)", "Max.Time(ms)",
                                 "Accuracy", "Precision", "Recall", "F1"]
        else:
            table.field_names = ["Planner", "Time(ms)", "Ave.Time(ms)", "Min.Time(ms)", "Max.Time(ms)", "Std.Time(ms)",
                                    "Tot.Check", "Tot.Reachable", "Tot.LegCheck",
                                    "Accuracy", "Precision", "Recall", "F1"]
        planner_names_correct = {"flt_cfg_planner_fast":"Ground Truth",
                                 "flt_cfg_planner":"KCFRC",
                                 "rrt_cfg_planner":"RRT-Connect",
                                 "minco_cfg_planner":"MINCO+LBFGS",
                                 "stomp_cfg_planner":"STOMP",
                                 "fec_planner":"FEC",}
        # Ground Truth Planner
        gt_planner = "flt_cfg_planner_fast"
        gt_reachable_demos = self.reachable_array[gt_planner]
        
        rc_perf = []
        for planner, demos in self.reachable_array.items():
            TP_sum = 0; FP_sum = 0; FN_sum = 0; TN_sum = 0
            for demo, reachable_list in demos.items():
                gt_reachable_arr = np.array(gt_reachable_demos[demo])
                reachable_arr = np.array(reachable_list)
                TP_sum += np.sum(np.logical_and(reachable_arr, gt_reachable_arr))
                FP_sum += np.sum(np.logical_and(reachable_arr, np.logical_not(gt_reachable_arr)))
                FN_sum += np.sum(np.logical_and(np.logical_not(reachable_arr), gt_reachable_arr))
                TN_sum += np.sum(np.logical_and(np.logical_not(reachable_arr), np.logical_not(gt_reachable_arr)))
            # Calculate Accuracy, Precision, Recall, F1
            accuracy = (TP_sum + TN_sum) / (TP_sum + FP_sum + FN_sum + TN_sum)
            precision = TP_sum / (TP_sum + FP_sum)
            recall = TP_sum / (TP_sum + FN_sum)
            f1 = 2 * precision * recall / (precision + recall)
            rc_perf.append([accuracy, precision, recall, f1])
        time_perf = []
        for planner, demos in self.benchmark_result.items():
            totcheck_num = 0    # points
            totreach_num = 0    # points
            totlegcheck_num = 0
            totcheck_time = 0
            totcheck_time2 = 0
            avecheck_time = 0       # per leg
            mincheck_time = math.inf # per leg
            maxcheck_time = 0       # per leg
            stdcheck_time = 0       # per leg
            for demo, benchmarks in demos.items():
                totcheck_num += benchmarks["RcTotCheckNum"]
                totreach_num += benchmarks["RcReachableNum"]
                totlegcheck_num += benchmarks["RcTotLegCheckNum"]
                totcheck_time += benchmarks["RcTotTime"]
                totcheck_time2 += benchmarks["RcAveTime2"] * benchmarks["RcTotLegCheckNum"]
                if benchmarks["RcMinTime"] < mincheck_time:
                    mincheck_time = benchmarks["RcMinTime"]
                if benchmarks["RcMaxTime"] > maxcheck_time:
                    maxcheck_time = benchmarks["RcMaxTime"]
            avecheck_time = totcheck_time / totlegcheck_num
            stdcheck_time = math.sqrt(max(totcheck_time2 / totlegcheck_num - avecheck_time**2, 0))
            time_perf.append([totcheck_time, avecheck_time, mincheck_time, maxcheck_time, stdcheck_time, 
                              totcheck_num, totreach_num, totlegcheck_num])
        if simplified:
            for i in range(planner_num):
                table.add_row([planner_names_correct[list(self.benchmark_result.keys())[i]]] + 
                                [time_perf[i][1], time_perf[i][3]] + rc_perf[i])
        else:
            for i in range(planner_num):
                table.add_row([planner_names_correct[list(self.benchmark_result.keys())[i]]] + 
                                time_perf[i]+ rc_perf[i])
        table.float_format = ".3"
        print("Total Check Num: ", totcheck_num, "\nTotal Leg Check Num: ", totlegcheck_num)
        return table
    
    def print_reachable(self):
        print(self.analysis_reachable())
        
    def save_reachable_csv(self, filename : str):
        table = self.analysis_reachable(simplified=True)
        table.float_format = ".3"
        with open (filename, "w") as f:
            string = table.get_csv_string()
            string = string.replace("\n", "")
            f.write(string)
            
        
        

In [11]:
AUTOBENCHMARK_DIR = os.path.join(ROOT_DIR, "data", "AutoBenchmarkOutput_20250131.json")
benchmark_result = json5.load(open(AUTOBENCHMARK_DIR, "r"))
analysis = OptBenchmarkAnalysis(benchmark_result)
# analysis.print_table("SuccessRate")
# analysis.print_table("Totaltime")
# analysis.print_table("AveTime")
# analysis.print_table("MaxTime")
# analysis.print_table("StdTime")
# analysis.print_table("AveLen")
# analysis.print_table("AveCtrl")
analysis.print_summary()
analysis.save_summary_csv(os.path.join(ROOT_DIR, "data", "benchmark_summary.csv"))

+---------------------+----------+-----------+-----------+------------+------------+
|       Planner       | Time(ms) | Len.(rad) |   Ctrl.   | Succ. Num. | Succ. Rate |
+---------------------+----------+-----------+-----------+------------+------------+
| Proposed(PITD)-mean |  0.820   |   2.541   |  201.632  |  304.000   |   0.984    |
|  Proposed(PITD)-max |  8.960   |   7.846   | 13792.500 |  304.000   |   0.984    |
|  Proposed(PITD)-std |  0.806   |   1.092   |  867.842  |  304.000   |   0.984    |
|                     |          |           |           |            |            |
|      STOMP-mean     |  1.212   |   1.585   |  390.881  |  303.000   |   0.981    |
|      STOMP-max      |  19.323  |   3.752   |  4290.860 |  303.000   |   0.981    |
|      STOMP-std      |  2.899   |   0.617   |  399.593  |  303.000   |   0.981    |
|                     |          |           |           |            |            |
|       FEC-mean      |  0.002   |   0.000   |   0.000   |  309.0

In [4]:
from matplotlib import rc
AUTOBENCHMARK_DIR = os.path.join(ROOT_DIR, "data", "AutoBenchmarkOutput_20250202.json")
REACHABLE_ARRAY_DIR = os.path.join(ROOT_DIR, "data", "AutoBenchmarkReachableArray_20250202.json")
benchmark_result = json5.load(open(AUTOBENCHMARK_DIR, "r"))
reachable_array = json5.load(open(REACHABLE_ARRAY_DIR, "r"))
rc_analysis = ReachableBenchmarkAnalysis(benchmark_result, reachable_array)
# rc_analysis.print_table("RcReachableNum")
# rc_analysis.print_table("RcTotTime")
# rc_analysis.print_table("RcAveTime")
rc_analysis.print_reachable()
rc_analysis.save_reachable_csv(os.path.join(ROOT_DIR, "data", "reachable_summary.csv"))

Total Check Num:  292800 
Total Leg Check Num:  732
+--------------+--------------+--------------+----------+-----------+--------+-------+
|   Planner    | Ave.Time(ms) | Max.Time(ms) | Accuracy | Precision | Recall |   F1  |
+--------------+--------------+--------------+----------+-----------+--------+-------+
| Ground Truth |    25.063    |    34.792    |  1.000   |   1.000   | 1.000  | 1.000 |
|    KCFRC     |    0.544     |    1.305     |  0.996   |   0.989   | 0.982  | 0.985 |
|    STOMP     |   152.785    |   1403.690   |  0.984   |   0.920   | 0.962  | 0.941 |
|     FEC      |    19.552    |    40.841    |  0.970   |   0.928   | 0.833  | 0.878 |
+--------------+--------------+--------------+----------+-----------+--------+-------+
Total Check Num:  292800 
Total Leg Check Num:  732
